![Project](https://img.shields.io/badge/NS%20Health%20%26%20Population%20Analytics-1B3A6B?style=for-the-badge&logoColor=white)

# 🏨 Top 10 Hospitalization Causes (MRDx) — ETL Pipeline
![Python](https://img.shields.io/badge/Python_3.12-C9A020?style=flat-square&logo=python&logoColor=white) ![pandas](https://img.shields.io/badge/pandas-1A6B5A?style=flat-square&logo=pandas&logoColor=white) ![Source](https://img.shields.io/badge/Source-CIHI-1B3A6B?style=flat-square)

> **Analytical Question:** How do NS hospitalization rates compare to the national average by condition type? What are the top 10 causes of hospitalization in NS vs. Canada?

This notebook extracts and cleans **CIHI Table 2 — Top 10 Inpatient Most Responsible Diagnoses (MRDx)** from 8 annual Excel workbooks (2017–18 through 2024–25) and stacks them into a single time-series dataset.

---

## 📦 Source Data

| Files | Content | Fiscal Years |
|-------|---------|-------------|
| `Hospitalization rates & others YYYY.xlsx` × 8 | Table 2: Top 10 inpatient MRDx by province | 2017–2018 to 2024–2025 |

**Source:** Canadian Institute for Health Information (CIHI) — Hospital Morbidity Database (HMDB)

---

## 🔧 ETL Pipeline Architecture

```
discover_files()
    └── for each .xlsx file:
          find_cihi_sheet()     ← content-based sheet detection
          read_cihi_table()     ← dynamic header row detection
          process_file()        ← clean + standardize + tag fiscal year
    └── pd.concat() → sort → export
```

---

## ⚠️ Known Data Challenges

| Challenge | Detail | Solution |
|-----------|--------|----------|
| Sheet name changes | 2017–2022: `'2. Top 10 inp MRDx'`; 2023+: `'Table 2'` | Content-based sheet detection |
| Header row position varies | Not always row 1 | Scan for row containing `'Province'` |
| Footnote symbols on province names | `Ont.†`, `Canada++` | Regex strip `[†‡*+§]+$` |
| Fiscal year not in data | Each workbook = one year | Extract from filename, add as column |

---


## Step 1 — Imports & Config


In [ ]:
import pandas as pd
from pathlib import Path
import re

DATA_DIR   = Path(r"C:\data\capstone\Q4")
OUTPUT_DIR = Path("clean")
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "hospitalization_causes.csv"


## Step 2 — Content-Based Sheet Detection

CIHI changed its sheet naming convention between fiscal years:
- 2017–2022: `'2. Top 10 inp MRDx'`
- 2023–2025: `'Table 2'`

Rather than hardcoding the sheet name (which would break on version changes), we **scan each sheet's content** for keywords that uniquely identify Table 2:
- `"most responsible diagnosis"` — confirms it's the MRDx table
- `"inpatient hospitalizations"` — confirms it's the hospitalization count table (not a rates table)


In [ ]:
def find_cihi_sheet(xls):
    """
    Finds CIHI Table 2 (Top 10 INP MRDx) using content-based detection.

    Reads the first 25 rows of each sheet and checks for identifying keywords.
    This is robust to sheet name changes across fiscal years.
    """
    for sheet in xls.sheet_names:
        try:
            preview = pd.read_excel(xls, sheet_name=sheet, nrows=25)
            text    = " ".join(preview.astype(str).values.flatten()).lower()

            if (
                "most responsible diagnosis" in text
                and "inpatient hospitalizations" in text
            ):
                return sheet

        except Exception:
            continue

    raise ValueError("CIHI Table 2 (Top 10 INP MRDx) not found in workbook")


## Step 3 — Dynamic Header Row Detection

CIHI Excel tables don't always start at row 1 — there are title rows, notes, and blank rows above the actual data header. We scan for the row containing `'Province'` and use that as our column header row.


In [ ]:
def read_cihi_table(path):
    """
    Reads a CIHI hospitalization Excel table with dynamic header detection.

    Steps:
    1. Find the correct sheet (content-based)
    2. Read with no header (raw mode)
    3. Find the row that contains 'Province' → that's the column header row
    4. Slice from that row onwards and promote to headers
    """
    xls   = pd.ExcelFile(path)
    sheet = find_cihi_sheet(xls)
    print(f"   ✅ Sheet used: {sheet}")

    # Read raw with no header — we'll detect the header row ourselves
    df = pd.read_excel(xls, sheet_name=sheet, header=None)

    # Find the first row that contains the word 'Province' (case-insensitive)
    header_row = df[
        df.apply(
            lambda r: r.astype(str).str.contains("province", case=False, na=False).any(),
            axis=1
        )
    ].index[0]

    # Slice: keep from header row onwards, promote row 0 to column names
    df = df.iloc[header_row:].reset_index(drop=True)
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)

    return df


## Step 4 — Process One Workbook

For each annual workbook:
1. Extract the fiscal year end from the filename (`'2018-2019'` → `2019`)
2. Read the table with dynamic header detection
3. Remove Excel-generated `Unnamed:` junk columns
4. Keep the first 6 columns (the Table 2 layout)
5. Standardize column names
6. Clean province footnote symbols (`Ont.†` → `Ont.`)
7. Cast numeric columns, drop rows with missing values
8. Tag with `Fiscal Year`


In [ ]:
def process_file(path, fiscal_year_range):
    print(f"   🧹 Processing {fiscal_year_range}")

    # Extract the end year: '2018-2019' → 2019
    fiscal_year = int(fiscal_year_range.split("-")[1])

    df = read_cihi_table(path)

    # ── Remove Excel junk columns ────────────────────────────
    df = df.loc[:, ~df.columns.astype(str).str.contains("unnamed", case=False)]

    # ── Keep first 6 real columns (Table 2 layout) ───────────
    df = df.iloc[:, :6]
    df.columns = [
        "Province/territory",
        "Rank",
        "Most responsible diagnosis",
        "Number of inpatient hospitalizations",
        "Percentage of inpatient hospitalizations",
        "Average acute length of stay",
    ]

    # ── Drop Rank (not useful for analysis) ──────────────────
    df = df.drop(columns=["Rank"])

    # ── Remove empty rows ────────────────────────────────────
    df = df[df["Province/territory"].notna()]

    # ── Clean footnote symbols from province names ───────────
    # Examples: 'Ont.†' → 'Ont.'  |  'Canada++' → 'Canada'
    df["Province/territory"] = (
        df["Province/territory"]
        .astype(str)
        .str.replace(r"[†‡*+§]+$", "", regex=True)
        .str.strip()
    )

    # ── Cast numeric columns ─────────────────────────────────
    for col in df.columns[2:]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna()

    # ── Tag with fiscal year ─────────────────────────────────
    df["Fiscal Year"] = fiscal_year

    return df


## Step 5 — Auto-Discover Files & Run Pipeline


In [ ]:
def discover_files():
    """
    Scans DATA_DIR for all CIHI hospitalization workbooks
    and extracts their fiscal year from the filename.
    Returns a sorted dict: {'2017-2018': Path(...), ...}
    """
    files = {}
    for f in DATA_DIR.glob("Hospitalization rates & others *.xlsx"):
        match = re.search(r"\d{4}-\d{4}", f.name)
        if match:
            files[match.group()] = f
    if not files:
        raise FileNotFoundError("No CIHI hospitalization files found in DATA_DIR")
    return dict(sorted(files.items()))


def run_pipeline():
    print("🚀 CIHI Table 2 – Top 10 INP MRDx ETL\n")

    files  = discover_files()
    frames = []

    for fy, path in files.items():
        frames.append(process_file(path, fy))

    final_df = pd.concat(frames, ignore_index=True)
    final_df = final_df.sort_values(
        ["Fiscal Year", "Province/territory", "Most responsible diagnosis"]
    )

    final_df.to_csv(OUTPUT_FILE, index=False)

    print("\n✅ ETL Complete")
    print(f"📁 Output : {OUTPUT_FILE}")
    print(f"📊 Rows   : {len(final_df)}")

    return final_df


final_df = run_pipeline()


## ✅ Output Summary

| Output | Location | Rows | Fiscal Years |
|--------|----------|------|-------------|
| Clean CSV | `./clean/hospitalization_causes.csv` | ~851 | 2017–18 to 2024–25 |

**Final schema:** `Province/territory` \| `Most responsible diagnosis` \| `Number of inpatient hospitalizations` \| `Percentage of inpatient hospitalizations` \| `Average acute length of stay` \| `Fiscal Year`

**Feeds into:**
- 📊 Power BI — Top 10 MRDx horizontal bar chart (NS vs. Canada)
- 📊 Power BI — Trend line: top conditions over 8 fiscal years

> ⚠️ **COVID note:** 2020–2021 data reflects pandemic impact — hospitalization volumes and condition mix are anomalous. Flag this year in all trend visuals.


---

*Nova Scotia Health & Population Analytics · CIHI Hospital Morbidity Database*
